In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans

In [2]:
df = pd.read_csv('../data/train.csv')


# Check shape, first rows, and basic info
print(df.shape)
print(df.head())
print(df.info())

(1000000, 10)
          id  vendor_id      pickup_datetime  passenger_count  \
0  id2793718          2  2016-06-08 07:36:19                1   
1  id3485529          2  2016-04-03 12:58:11                1   
2  id1816614          2  2016-06-05 02:49:13                5   
3  id1050851          2  2016-05-05 17:18:27                2   
4  id0140657          1  2016-05-12 17:43:38                4   

   pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
0        -73.985611        40.735943         -73.980331         40.760468   
1        -73.978394        40.764351         -73.991623         40.749859   
2        -73.989059        40.744389         -73.973381         40.748692   
3        -73.990326        40.731136         -73.991264         40.748917   
4        -73.789497        40.646675         -73.987137         40.759232   

  store_and_fwd_flag  trip_duration  
0                  N           1040  
1                  N            827  
2                 

In [3]:
!pip install holidays


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import holidays

us_holidays = holidays.US(years=list(range(2016,2050)), state='NY')  


df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['pickup_date'] = df['pickup_datetime'].dt.date
df['is_holiday'] = df['pickup_datetime'].dt.date.isin(us_holidays).astype(int)  
df['dayofweek'] = df.pickup_datetime.dt.dayofweek
df['month'] = df.pickup_datetime.dt.month
df['hour'] = df.pickup_datetime.dt.hour
df['day'] = df.pickup_datetime.dt.day

In [5]:
df['rush_hours'] = ((df['hour'].between(8,10)) | (df['hour'].between(17,21))).astype(int)
df["night_flag"] = np.where((df["hour"] >= 21) | (df["hour"] < 5), 1, 0)
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

In [6]:
# Identify and Handle Anomalies:
daily_trip_count = df.groupby('pickup_date').size()
z_scores = (daily_trip_count - daily_trip_count.mean()) / daily_trip_count.std()
anomalous_days = z_scores[abs(z_scores) > 2].index

df['is_anomaly'] = df['pickup_date'].apply(lambda x: 1 if x in anomalous_days else 0)

In [7]:

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers
    between two points on the earth (specified in decimal degrees).
    """
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # haversine formula
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371  # Radius of earth in kilometers
    return c * r

In [8]:
# Apply to dataset
df['haversine_distance'] = haversine(
    df['pickup_longitude'], df['pickup_latitude'],
    df['dropoff_longitude'], df['dropoff_latitude']
)


In [9]:
df["manhattan_distance"] = (
    np.abs(df["dropoff_longitude"] - df["pickup_longitude"]) +
    np.abs(df["dropoff_latitude"] - df["pickup_latitude"])
)

In [10]:
def bearing(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


In [11]:
df["bearing"] = bearing(
    df["pickup_longitude"], df["pickup_latitude"],
    df["dropoff_longitude"], df["dropoff_latitude"]
)

### Traffic Congestion Analysis

This cell quantifies traffic patterns by comparing individual trip speeds against historical averages for specific time slots.

1.  **Feature Engineering (`speed`):** Calculates the trip speed in distance per hour.
2.  **Temporal Baselining:** Uses `groupby` and `transform` to calculate the **mean** and **standard deviation** of speeds for every unique combination of `dayofweek` and `hour`. 
3.  **Anomalous Speed Detection (`z_score`):** Determines how far a trip's speed deviates from the norm. A negative Z-score indicates a trip was slower than average.
4.  **Congestion Labeling:** Identifies "Congested" trips where the speed is more than one standard deviation below the mean ($Z < -1$).
5.  **Probability Mapping (`congestion_rate`):** Calculates the likelihood of congestion for each time slot (e.g., "Monday at 8 AM") by averaging the binary congestion flags.
6.  **Cleanup:** Drops intermediate statistical columns to keep the DataFrame lean, retaining only the final `congestion_rate` feature.

In [12]:
# Analyze Traffic Congestion:
df['speed'] = df['haversine_distance'] / (df['trip_duration'] / 3600)
df['mean_speed'] = df.groupby(['dayofweek','hour'])['speed'].transform('mean')
df['std_speed']  = df.groupby(['dayofweek','hour'])['speed'].transform('std')

df['z_score'] = (df['speed'] - df['mean_speed']) / df['std_speed']

df['traffic_congestion'] = (df['z_score'] < -1).astype(int)

df['congestion_rate'] = df.groupby(['dayofweek','hour'])['traffic_congestion'].transform('mean')
df.drop(columns=['traffic_congestion','z_score','std_speed','mean_speed'], inplace=True)

In [13]:
# Airport Proximity Features: 
airports = {
    'JFK': (-73.7781, 40.6413),   # lon, lat
    'LGA': (-73.8740, 40.7769),   # lon, lat
    'EWR': (-74.1745, 40.6895)    # lon, lat
}

threshold_distance = 1  # 1 km

def is_near_airport(lon, lat, airports=airports, threshold=1):
    for airport_coords in airports.values():
        if haversine(lon, lat, *airport_coords) <= threshold:
            return True
    return False


df['is_pickup_at_airport'] = df.apply(lambda row: is_near_airport(row['pickup_longitude'], row['pickup_latitude'], airports, threshold_distance), axis=1).astype(int)
df['is_dropoff_at_airport'] = df.apply(lambda row: is_near_airport(row['dropoff_longitude'], row['dropoff_latitude'], airports, threshold_distance), axis=1).astype(int)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 27 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   id                     1000000 non-null  object        
 1   vendor_id              1000000 non-null  int64         
 2   pickup_datetime        1000000 non-null  datetime64[ns]
 3   passenger_count        1000000 non-null  int64         
 4   pickup_longitude       1000000 non-null  float64       
 5   pickup_latitude        1000000 non-null  float64       
 6   dropoff_longitude      1000000 non-null  float64       
 7   dropoff_latitude       1000000 non-null  float64       
 8   store_and_fwd_flag     1000000 non-null  object        
 9   trip_duration          1000000 non-null  int64         
 10  pickup_date            1000000 non-null  object        
 11  is_holiday             1000000 non-null  int64         
 12  dayofweek              100000

In [15]:
df = df.drop(columns=["id"])  # Assuming 'id' is not needed for modeling
df = df.drop(columns=["pickup_datetime"])

print("After dropping:", df.shape)


After dropping: (1000000, 25)


In [16]:
df["log_trip_duration"] = np.log1p(df["trip_duration"])
print (df.head(10))

   vendor_id  passenger_count  pickup_longitude  pickup_latitude  \
0          2                1        -73.985611        40.735943   
1          2                1        -73.978394        40.764351   
2          2                5        -73.989059        40.744389   
3          2                2        -73.990326        40.731136   
4          1                4        -73.789497        40.646675   
5          2                3        -73.969833        40.768570   
6          2                1        -73.988419        40.760006   
7          1                1        -73.987854        40.749695   
8          2                1        -73.955017        40.764462   
9          1                1        -73.971535        40.794827   

   dropoff_longitude  dropoff_latitude store_and_fwd_flag  trip_duration  \
0         -73.980331         40.760468                  N           1040   
1         -73.991623         40.749859                  N            827   
2         -73.973381   

In [17]:
def sin_cos_encode(values, period):
    return np.sin(2 * np.pi * values / period), np.cos(2 * np.pi * values / period)

In [18]:
df['hour_x'], df['hour_y'] = sin_cos_encode(df['hour'], 24)
df['dayofweek_x'], df['dayofweek_y'] = sin_cos_encode(df['dayofweek'], 7)
df['month_x'], df['month_y'] = sin_cos_encode(df['month'], 12)
df['day_x'], df['day_y'] = sin_cos_encode(df['day'], 31)

## **Spatial & Corridor Feature Summary**

This block of code implements a **hierarchical spatial encoding strategy**. It moves from raw coordinates to neighborhood-level behaviors, and finally to specific route (corridor) analysis.

---

### **1. Neighborhood Clustering (`MiniBatchKMeans`)**

* **The Logic:** Uses the coordinates to partition NYC into **20 discrete clusters** (neighborhoods) for both pickups and dropoffs.
* **Benefit:** Reduces millions of granular GPS points into high-level categorical "zones," making it easier for the model to learn localized traffic patterns.
* **Performance:** `MiniBatchKMeans` is used to maintain high speed on the 1-million-row dataset by processing data in small batches.

### **2. Zonal Intensity Features**

* **Pickup Intensity:** Calculates the volume (`trip_ct`), typical time (`median_trip`), and flow efficiency (`median_speed`) for each starting neighborhood.
* **Dropoff Intensity:** Performs the same for destination neighborhoods.
* **Benefit:** Tells the model if a trip is starting or ending in a "hotspot" (like Midtown) or a "slow zone," providing context about congestion at both ends of the journey.

### **3. The "Golden Feature": Corridor Stats**

* **The Logic:** Aggregates the `median_trip_duration` for every unique **Pickup-to-Dropoff pair** (e.g., Cluster 5 $\rightarrow$ Cluster 12).
* **Benefit:** This is the most predictive spatial feature. It captures the "historical typical time" for specific routes, accounting for physical barriers like bridges or tunnels that distance formulas (Haversine/Manhattan) often ignore.

---

### **Final Output Columns**

| Feature | Description |
| --- | --- |
| `pickup_cluster` / `dropoff_cluster` | The neighborhood ID (0–19). |
| **`cluster_trip_ct`** | Popularity/Density of the neighborhood. |
| **`cluster_median_speed`** | Average traffic flow efficiency of the zone. |
| **`corridor_median_duration`** | Historical median time for that specific route. |

In [ ]:
def build_spatial_clusters(data, lat_col, lon_col, n_clusters=20, sample_size=200_000, random_state=42):
    coords = data[[lat_col, lon_col]].dropna()
    if coords.empty:
        return pd.Series(-1, index=data.index, dtype=int)
    sample = coords.sample(min(sample_size, len(coords)), random_state=random_state)
    model = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10_000, random_state=random_state)
    model.fit(sample)
    labels = model.predict(coords)
    full_labels = pd.Series(-1, index=data.index, dtype=int)
    full_labels.loc[coords.index] = labels
    return full_labels

print("Fitting MiniBatchKMeans clusters ...")
df['pickup_cluster'] = build_spatial_clusters(df, 'pickup_latitude', 'pickup_longitude')
df['dropoff_cluster'] = build_spatial_clusters(df, 'dropoff_latitude', 'dropoff_longitude')

pickup_intensity = (
    df.groupby('pickup_cluster')
      .agg(
          pickup_cluster_trip_ct=('trip_duration', 'size'),
          pickup_cluster_median_trip=('trip_duration', 'median'),
          pickup_cluster_median_speed=('speed', 'median')
      )
)
df = df.merge(pickup_intensity, left_on='pickup_cluster', right_index=True, how='left')
dropoff_intensity = (
    df.groupby('dropoff_cluster')
      .agg(
          dropoff_cluster_trip_ct=('trip_duration', 'size'),
          dropoff_cluster_median_trip=('trip_duration', 'median'),
          dropoff_cluster_median_speed=('speed', 'median')
      )
)
df = df.merge(dropoff_intensity, left_on='dropoff_cluster', right_index=True, how='left')


corridor_stats = (
    df.groupby(['pickup_cluster', 'dropoff_cluster'])['trip_duration']
      .median()
      .rename('corridor_median_duration')
      .reset_index()
)
df = df.merge(corridor_stats, on=['pickup_cluster', 'dropoff_cluster'], how='left')


Fitting MiniBatchKMeans clusters ...


## **Advanced Feature Engineering & Aggregation Summary**

This block focuses on creating **engineered ratios**, **normalized temporal metrics**, and **behavioral benchmarks**. These features help the model understand relative performance rather than just raw numbers.

---

### **1. Engineered Ratios & Geometric Scaling**

* **`distance_per_passenger`**: A proxy for trip efficiency and passenger load. Clipping at **1** ensures no division-by-zero errors.
* **`log_distance`**: Applies a logarithmic transformation ($\ln(1+x)$) to distance. This "squashes" the long tail of outlier trips, making the distribution more normal and easier for models to process.

---

### **2. Temporal "Heatbeat" Aggregations**

* **`week_hour`**: Creates a continuous index (0–167) for every hour in a week.
* **`week_hour_trip_ct_norm`**: Normalizes trip volume (0 to 1). This tells the model how "busy" the city is relative to its peak capacity, regardless of the absolute numbers.
* **`daily_stats`**: Captures the **median** and **95th percentile** duration for each specific day. The P95 is particularly useful for identifying "bad traffic days" where even standard trips took much longer than usual.

---

### **3. Performance Benchmarking**

* **`vendor_hour_speed_median`**: Profiles the typical speed of different taxi fleets at specific times, capturing operational differences between vendors.
* **`speed_minus_cluster_median`**: A **Relative Efficiency** feature. It measures how a trip performed compared to its local neighborhood average. This helps isolate whether a slow trip was due to local neighborhood congestion or other factors.

---

### **4. Interaction Flags**

* **`weekend_rush`**: Specifically targets Saturday/Sunday peak windows, which often have different traffic drivers (leisure/tourism) compared to weekday work commutes.
* **`night_weekday_flag`**: Isolates "Free-Flow" periods (weekday nights) where the model should expect the highest speeds and shortest durations for a given distance.

---

### **Output Preview**

The resulting dataframe now includes high-context features like:

* **Trip Density** (Volume relative to peak)
* **Relative Speed** (Performance vs. local neighborhood)
* **Fleet Profiles** (Vendor-specific temporal behavior)

In [ ]:
df['distance_per_passenger'] = df['haversine_distance'] / df['passenger_count'].clip(lower=1)
df['log_distance'] = np.log1p(df['haversine_distance'])
df['week_hour'] = df['dayofweek'] * 24 + df['hour']

hourly_features = (
    df.groupby('week_hour')['trip_duration']
      .agg(week_hour_trip_ct='size', week_hour_median_duration='median')
      .reset_index()
)
df = df.merge(hourly_features, on='week_hour', how='left')
df['week_hour_trip_ct_norm'] = df['week_hour_trip_ct'] / df['week_hour_trip_ct'].max()

daily_stats = (
    df.groupby('pickup_date')['trip_duration']
      .agg(daily_median_duration='median', daily_p95_duration=lambda x: x.quantile(0.95))
)
df = df.merge(daily_stats, left_on='pickup_date', right_index=True, how='left')

vendor_hour_speed = df.groupby(['vendor_id', 'hour'])['speed'].transform('median')
df['vendor_hour_speed_median'] = vendor_hour_speed
df['pickup_cluster_median_speed'] = df['pickup_cluster_median_speed'].fillna(df['pickup_cluster_median_speed'].median())
df['speed_minus_cluster_median'] = df['speed'] - df['pickup_cluster_median_speed']

df['weekend_rush'] = df['is_weekend'] * df['rush_hours']
df['night_weekday_flag'] = ((1 - df['is_weekend']) & (df['night_flag'] == 1)).astype(int)

df[['distance_per_passenger', 'week_hour', 'week_hour_trip_ct', 'vendor_hour_speed_median']].head()


,distance_per_passenger,week_hour,week_hour_trip_ct,vendor_hour_speed_median
0,2.763050,55,7135,14.247339
1,1.959178,156,7293,10.799554
2,0.280954,146,5981,17.458954
3,0.989330,89,7279,11.579567
4,5.209436,89,7279,11.384158


**Temporal/Distance Findings**: `week_hour_trip_ct_norm` clearly separates weekday commute spikes (values >0.8) from quieter overnight periods, while `distance_per_passenger` exposes under-utilized rides (solo riders on 10+ km trips). These signals help tree models split on demand pressure as well as ride efficiency.


In [21]:
from sklearn.preprocessing import OneHotEncoder

cols_to_encode = ['vendor_id', 'passenger_count']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  
encoder.fit(df[cols_to_encode])

df_encoded_array = encoder.transform(df[cols_to_encode])
df_encoded_df = pd.DataFrame(df_encoded_array, columns=encoder.get_feature_names_out(cols_to_encode))
df = pd.concat([df.drop(columns=cols_to_encode), df_encoded_df], axis=1)

In [22]:
columns = ['trip_duration', 'haversine_distance', 'manhattan_distance']

factor = 2.5

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

trip_duration: 21507 outliers (2.15%)
haversine_distance: 54997 outliers (5.50%)
manhattan_distance: 57010 outliers (5.70%)


**Outlier Filtering Impact**: Applying a 2.5×IQR cap on `trip_duration`, `haversine_distance`, and `manhattan_distance` keeps ≈93% of NYC-filtered trips (932k of 997k rows) while stripping implausible multi-day rides and cross-country coordinates. This stabilizes downstream scaling and prevents tree models from dedicating depth to noise.


In [23]:
columns = ['trip_duration', 'haversine_distance', 'manhattan_distance']

factor = 2.5

df_clean = df.copy()
original_rows = len(df_clean)

for col in columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]

cleaned_rows = len(df_clean)
deleted_rows = original_rows - cleaned_rows
deleted_percent = deleted_rows / original_rows * 100



print(f"Original rows: {original_rows}")
print(f"Cleaned rows: {cleaned_rows}")
print(f"Deleted rows: {deleted_rows} ({deleted_percent:.2f}%)")

Original rows: 1000000
Cleaned rows: 911001
Deleted rows: 88999 (8.90%)


In [24]:
bool_cols = df_clean.select_dtypes(include='bool').columns

df_clean[bool_cols] = df_clean[bool_cols].astype(int)

In [25]:
df_clean.head()
print(df_clean.shape)
print(df_clean.info())
df_clean.describe()

(911001, 60)
<class 'pandas.core.frame.DataFrame'>
Index: 911001 entries, 0 to 999999
Data columns (total 60 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   pickup_longitude             911001 non-null  float64
 1   pickup_latitude              911001 non-null  float64
 2   dropoff_longitude            911001 non-null  float64
 3   dropoff_latitude             911001 non-null  float64
 4   store_and_fwd_flag           911001 non-null  object 
 5   trip_duration                911001 non-null  int64  
 6   pickup_date                  911001 non-null  object 
 7   is_holiday                   911001 non-null  int64  
 8   dayofweek                    911001 non-null  int32  
 9   month                        911001 non-null  int32  
 10  hour                         911001 non-null  int32  
 11  day                          911001 non-null  int32  
 12  rush_hours                   911001 non-null  int6

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,is_holiday,dayofweek,month,hour,day,...,vendor_id_1,vendor_id_2,passenger_count_0,passenger_count_1,passenger_count_2,passenger_count_3,passenger_count_4,passenger_count_5,passenger_count_6,passenger_count_7
count,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,...,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000,911001.000000
mean,-73.979296,40.752875,-73.977316,40.752936,707.784793,0.024733,3.056014,3.507204,13.643796,15.487338,...,0.467288,0.532712,0.000040,0.711215,0.142096,0.040875,0.019279,0.053272,0.033221,0.000002
std,0.059406,0.027138,0.059920,0.029915,446.188507,0.155311,1.950718,1.680482,6.366512,8.699575,...,0.498929,0.498929,0.006286,0.453198,0.349149,0.198000,0.137503,0.224576,0.179212,0.001482
min,-121.933342,34.359695,-121.933304,34.359695,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-73.992172,40.738506,-73.991608,40.737671,375.000000,0.000000,1.000000,2.000000,9.000000,8.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-73.982170,40.754364,-73.980583,40.754799,609.000000,0.000000,3.000000,4.000000,14.000000,15.000000,...,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,-73.969437,40.767937,-73.965996,40.769402,940.000000,0.000000,5.000000,5.000000,19.000000,23.000000,...,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,-61.335529,43.911762,-61.335529,43.911762,2766.000000,1.000000,6.000000,6.000000,23.000000,31.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [26]:
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})

In [38]:
# create directory if it does not exist
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv('../data/processed/train_processed.csv', index=False)